# Cell marker positivity — training

A CNN that decides whether one segmented cell is positive for one IMC marker.

The input to the network is a crop centred on the cell:

* the target marker channel plus its neighbouring metal channels (`N_SIDE` on each side),
* the binary mask of the target cell,
* optionally the binary mask of all *other* cells in the crop (`USE_OTHER_CELLS`).

Nothing is written to disk. Only the marker volumes and one instance-label map per
region are held in memory (~90 MB in total); the crops are cut out of those volumes
on the fly, every iteration.

In [ ]:
import numpy as np
import nibabel as nib
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy import ndimage
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

# --- data -----------------------------------------------------------------
DATA_DIR = "TrainData"

DATASETS = [
    ("X02_a0_full.nii.gz", "X02_a0_full_mask_modified.nii.gz"),
    ("EMMA1.nii", "EMMA1_inference_corrected.nii"),
    ("20231201_713Lung_E_3_20231201-154853-900840_Q013_markers_channels_zstack.nii.gz",
     "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_positivity_corrected.nii.gz"),
    ("X02_a0_full.nii.gz", "X02_a0_full_mask_modified_eroded.nii.gz"),
    ("EMMA1.nii", "EMMA1_inference_corrected_eroded.nii"),
    ("20231201_713Lung_E_3_20231201-154853-900840_Q013_markers_channels_zstack.nii.gz",
     "20231201_713Lung_E_3_20231201-154853-900840_Q013_cell_positivity_corrected_dilated.nii.gz"),
]

# Values used in the positivity masks.
NEGATIVE, POSITIVE, UNKNOWN = 1, 2, 3

# --- model input ----------------------------------------------------------
CROP = 64
N_SIDE = 2                 # neighbouring metal channels on each side
USE_OTHER_CELLS = True    # add the mask of the surrounding cells

N_MARKERS = 2 * N_SIDE + 1
N_CHANNELS = N_MARKERS + 1 + int(USE_OTHER_CELLS)

# --- training -------------------------------------------------------------
SEED = 42
VAL_FRAC = 0.2
BATCH_SIZE = 512
EPOCHS = 100
BASE_LR, MIN_LR, CYCLE_LENGTH = 1e-3, 1e-6, 20

print(f"Input: {CROP}x{CROP}x{N_CHANNELS}")

## Load the regions and index the labelled cells

Each region is a marker stack `(H, W, Z)` and a positivity mask of the same shape,
in which every cell is filled with 1 (negative), 2 (positive) or 3 (unknown) for
each marker channel.

The cell segmentation is the same for every marker, so the cells are labelled once
per region. Every cell is uniform within a channel, so one representative pixel per
cell is enough to read off its label for all channels at once. Cells that are
unknown in a channel are simply left out of the index.

In [ ]:
class Region:
    """One imaged region held in memory."""

    def __init__(self, markers, cells):
        self.markers = markers   # (H, W, Z) float32, the marker stack
        self.cells = cells       # (H, W)    int32, cell instance labels


def load_region(image_file, mask_file):
    """Return the region and, per cell and channel, its positivity value."""
    markers = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{image_file}"))
    positivity = nib.as_closest_canonical(nib.load(f"{DATA_DIR}/{mask_file}"))

    markers = markers.get_fdata().astype(np.float32)
    positivity = positivity.get_fdata().astype(np.int16)

    if markers.shape != positivity.shape:
        raise ValueError(f"shape mismatch: {markers.shape} vs {positivity.shape}")

    cells, n_cells = ndimage.label(positivity[:, :, 0] > 0)
    cells = cells.astype(np.int32)

    # One pixel per cell, then read that pixel in every channel: (n_cells, Z).
    representative = np.zeros(n_cells + 1, dtype=np.int64)
    representative[cells.ravel()] = np.arange(cells.size)
    values = positivity.reshape(-1, positivity.shape[2])[representative[1:]]

    centres = ndimage.center_of_mass(cells > 0, cells, np.arange(1, n_cells + 1))
    centres = np.array(centres).astype(np.int32)

    return Region(markers, cells), values, centres


regions = []
index = []    # columns: region, z, cell id, centre y, centre x
labels = []   # 1.0 = positive

for region_id, (image_file, mask_file) in enumerate(DATASETS):
    region, values, centres = load_region(image_file, mask_file)
    regions.append(region)

    labelled = (values == NEGATIVE) | (values == POSITIVE)
    cell, z = np.nonzero(labelled)

    index.append(np.column_stack([
        np.full(cell.size, region_id), z, cell + 1, centres[cell]
    ]))
    labels.append(values[cell, z] == POSITIVE)

    print(f"{image_file[:40]:40s} {region.markers.shape}  "
          f"{values.shape[0]:5d} cells  {cell.size:6d} labelled  "
          f"{labelled.size - cell.size:5d} unknown")

index = np.concatenate(index).astype(np.int32)
labels = np.concatenate(labels).astype(np.float32)

memory = sum(r.markers.nbytes + r.cells.nbytes for r in regions)
print(f"\nTotal: {len(index)} samples, {labels.mean():.1%} positive, "
      f"{memory / 1e6:.0f} MB in memory")

## Cutting a sample out of a region

The crop is centred on the cell but kept inside the image, exactly as the inference
notebook does it.

Near the ends of the stack some of the neighbouring channels do not exist. Those are
left empty rather than repeating the edge channel: a copy of the target channel would
be perfectly correlated with it, which is exactly what spillover looks like, whereas
an empty channel simply says that there is no such neighbour.

Samples are augmented by rotating and flipping the crop, and by reversing the order
of the marker channels. The reversal makes the model indifferent to whether the stack
was written in ascending or descending order; the target channel sits in the middle,
so only its neighbours swap sides. The two mask channels keep their place.

In [ ]:
def make_sample(region_id, z, cell_id, centre_y, centre_x):
    region = regions[region_id]
    height, width, n_z = region.markers.shape
    half = CROP // 2

    y = min(max(centre_y - half, 0), height - CROP)
    x = min(max(centre_x - half, 0), width - CROP)

    cells = region.cells[y:y + CROP, x:x + CROP]

    # Channels beyond either end of the stack do not exist and stay empty.
    wanted = np.arange(z - N_SIDE, z + N_SIDE + 1)
    exists = (wanted >= 0) & (wanted < n_z)

    markers = np.zeros((CROP, CROP, N_MARKERS), dtype=np.float32)
    markers[..., exists] = region.markers[y:y + CROP, x:x + CROP, wanted[exists]]

    planes = [markers, (cells == cell_id)[..., None]]
    if USE_OTHER_CELLS:
        planes.append(((cells > 0) & (cells != cell_id))[..., None])

    return np.concatenate(planes, axis=-1).astype(np.float32)


def augment(sample, rng):
    """Random quarter turn, random flip, and a random reversal of the channel order."""
    sample = np.rot90(sample, rng.integers(4), axes=(0, 1))
    if rng.random() < 0.5:
        sample = sample[:, ::-1]
    if rng.random() < 0.5:
        sample = np.concatenate(
            [sample[..., N_MARKERS - 1::-1], sample[..., N_MARKERS:]], axis=-1)
    return sample


class CellSequence(tf.keras.utils.Sequence):
    def __init__(self, rows, shuffle, augment_samples):
        self.rows = rows
        self.shuffle = shuffle
        self.augment_samples = augment_samples
        self.rng = np.random.default_rng(SEED)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.rows) / BATCH_SIZE))

    def on_epoch_end(self):
        self.order = self.rng.permutation(self.rows) if self.shuffle else self.rows

    def __getitem__(self, batch):
        rows = self.order[batch * BATCH_SIZE:(batch + 1) * BATCH_SIZE]
        x = np.empty((len(rows), CROP, CROP, N_CHANNELS), dtype=np.float32)

        for i, row in enumerate(rows):
            sample = make_sample(*index[row])
            x[i] = augment(sample, self.rng) if self.augment_samples else sample

        return x, labels[rows]


shuffled = np.random.default_rng(SEED).permutation(len(index))
n_val = int(len(index) * VAL_FRAC)

train_gen = CellSequence(shuffled[n_val:], shuffle=True, augment_samples=True)
val_gen = CellSequence(shuffled[:n_val], shuffle=False, augment_samples=False)

print(f"Train: {len(train_gen.rows)} samples, val: {len(val_gen.rows)} samples")

## Check a sample

Every input channel of one random cell, so it is easy to see that the marker
channels, the cell mask and the surrounding cells line up.

In [ ]:
row = np.random.default_rng(0).integers(len(index))
sample = make_sample(*index[row])

names = [f"marker z{z:+d}" for z in range(-N_SIDE, N_SIDE + 1)] + ["this cell"]
if USE_OTHER_CELLS:
    names.append("other cells")

fig, axes = plt.subplots(1, N_CHANNELS, figsize=(3 * N_CHANNELS, 3.4))
for ax, name, plane in zip(axes, names, np.moveaxis(sample, -1, 0)):
    ax.imshow(plane, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

region_id, z, cell_id = index[row, :3]
fig.suptitle(f"region {region_id}, channel {z}, cell {cell_id} — "
             f"{'POSITIVE' if labels[row] else 'NEGATIVE'}")
plt.tight_layout()
plt.show()

## Model

In [ ]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(filters, kernel_size, strides=strides,
                      padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)


def make_marker_cnn(input_shape):
    inputs = layers.Input(shape=input_shape)

    c = 32
    x = conv_bn_relu(inputs, c)
    x = conv_bn_relu(x, c)
    x = conv_bn_relu(x, c)
    x = conv_bn_relu(x, c)

    x = conv_bn_relu(x, 2 * c, strides=2)
    x = conv_bn_relu(x, 4 * c, strides=2)
    x = conv_bn_relu(x, 8 * c, strides=2)

    x = layers.Conv2D(1, 1)(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Activation("sigmoid")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


model = make_marker_cnn((CROP, CROP, N_CHANNELS))
model.summary()

## Train

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)


def cosine_annealing_with_restarts(epoch, lr):
    t = (epoch % CYCLE_LENGTH) / CYCLE_LENGTH
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + np.cos(np.pi * t))


callbacks = [
    tf.keras.callbacks.LearningRateScheduler(cosine_annealing_with_restarts),
    tf.keras.callbacks.CSVLogger("training_log.csv", append=True),
    tf.keras.callbacks.ModelCheckpoint("checkpoints/marker_cnn_epoch_{epoch:03d}.h5"),
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    workers=8,
    use_multiprocessing=False,
)

model.save("marker_cnn_final.h5")
print("Training complete.")

## Training curves

In [ ]:
for key in ["loss", "accuracy", "auc"]:
    plt.figure()
    plt.plot(history.history[key], label=f"train {key}")
    plt.plot(history.history[f"val_{key}"], label=f"val {key}")
    plt.xlabel("epoch")
    plt.ylabel(key)
    plt.legend()
    plt.grid(True)
    plt.show()

## Architecture figure (optional)

Needs `pip install visualkeras pillow`.

In [ ]:
import visualkeras
from PIL import Image
from IPython.display import Image as IPyImage


def text_callable(layer_index, layer):
    shape = [s for s in layer.output.shape if s is not None]
    text = "x".join(str(s) for s in shape[:-1]) + f"\n{shape[-1]}" if shape else "?"
    return f"{text}\n{layer.name}", bool(layer_index % 2)


image = visualkeras.layered_view(
    model, legend=True, show_dimension=True, spacing=15,
    draw_volume=True, scale_xy=0.8, text_callable=text_callable,
)
image = image.resize((int(image.width * 0.6), image.height), Image.BILINEAR)
image.save("marker_cnn_3d.png")

IPyImage("marker_cnn_3d.png")